In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gdabhishek/fertilizer-prediction")

print("Path to dataset files:", path)

100%|██████████| 1.27k/1.27k [00:00<00:00, 2.52MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/gdabhishek/fertilizer-prediction/versions/1


**1. Environment Setup & Data Loading:**

As per the journal, this module requires input features including NPK levels, crop type, and growth stage.

In [4]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
import os # Import the os module

# Load your dataset
# Requirement: Columns for Nitrogen (N), Phosphorous (P), Potassium (K),
# Crop_Type, Growth_Stage, and the target Dosage_kg_ha.
df = pd.read_csv(os.path.join(path, 'Fertilizer Prediction.csv')) # Use the 'path' variable to locate the downloaded file

**2. Data Preprocessing (Step 3: NLP/Normalization):**

While the journal uses NLP for user queries, for structured model training, we must normalize and encode categorical labels like crop types and growth stages.

In [7]:
# Encoding categorical variables for the ML model
le_crop = LabelEncoder()
df['Crop Type'] = le_crop.fit_transform(df['Crop Type'])

le_soil = LabelEncoder()
df['Soil Type'] = le_soil.fit_transform(df['Soil Type'])

# Define Features (X) and Target (y)
# Note: In this specific dataset 'Temparature' and 'Humidty' are often misspelled.
# We will use the actual column names available in the dataframe.
# Common columns: 'Nitrogen', 'Phosphorous', 'Potassium', 'Temparature', 'Humidty', 'Moisture', 'Soil Type', 'Crop Type'

# Map correct column names from the actual dataframe keys
actual_cols = df.columns.tolist()
feature_cols = []
for col in ['Nitrogen', 'Phosphorous', 'Potassium', 'Temparature', 'Humidty', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type']:
    if col in actual_cols:
        feature_cols.append(col)

X = df[feature_cols]
y = df['Dosage_kg_ha'] if 'Dosage_kg_ha' in df.columns else df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**3. Model Training: Decision Support (Step 5):**

The journal recommends using a Decision Tree or Gradient Boosting Regression for this module because they handle non-linear relationships and structured numeric data effectively.

In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Option A: Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Option B: Gradient Boosting Classifier (Recommended for higher precision)
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

# Accuracy Check
print(f"Decision Tree Accuracy: {dt_model.score(X_test, y_test)*100:.2f}%")
print(f"Gradient Boosting Accuracy: {gb_model.score(X_test, y_test)*100:.2f}%")

Decision Tree Accuracy: 100.00%
Gradient Boosting Accuracy: 95.00%


**4. Exporting the Module:**

We export the model and encoders so they can be called by the RAG pipeline to assemble natural, farmer-friendly answers.

In [11]:
# Save model for integration into the AgriBot backend
with open('fertilizer_model.pkl', 'wb') as f:
    pickle.dump(gb_model, f)

# Save encoders for consistent real-time query processing
pickle.dump(le_crop, open('le_crop_fert.pkl', 'wb'))
pickle.dump(le_soil, open('le_soil_fert.pkl', 'wb'))

**📊 Module: Performance Analysis & Visualization**

Goal: Evaluate the model's accuracy, visualize the results, and simulate response time metrics as defined in the research.

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import time
import pickle

# 1. Load your trained model and test data
# (Assuming X_test and y_test are still in your environment)
model = pickle.load(open('fertilizer_model.pkl', 'rb'))
# Note: le_fert is not needed as the target 'y' was kept as strings/categories

# 2. Generate Predictions
start_time = time.time()
y_pred = model.predict(X_test)
end_time = time.time()

# 3. Calculate Core Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%") # Target per journal: ~89.7%
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Accuracy: 95.00%

Classification Report:
               precision    recall  f1-score   support

    10-26-26       1.00      1.00      1.00         2
    14-35-14       0.50      1.00      0.67         1
       20-20       1.00      1.00      1.00         1
       28-28       1.00      1.00      1.00         5
         DAP       1.00      0.80      0.89         5
        Urea       1.00      1.00      1.00         6

    accuracy                           0.95        20
   macro avg       0.92      0.97      0.93        20
weighted avg       0.97      0.95      0.96        20



**💾 Module: Saving to Google Drive:**

Goal: Mount Google Drive and move the exported files to a dedicated project folder for long-term storage.

In [14]:
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
# This allows the notebook to access your personal Drive storage
drive.mount('/content/drive')

# 2. Define the Destination Path
# You can change 'AgriBot_Project' to your specific folder name
drive_path = '/content/drive/MyDrive/AgriBot_Project/Models/'

# Create the directory if it doesn't exist
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"Created new directory: {drive_path}")

# 3. List of files to save
# These are the quantitative engine files generated in the previous steps
files_to_move = [
    'fertilizer_model.pkl',
    'le_crop_fert.pkl',
    'le_stage_fert.pkl'
]

# 4. Move Files to Drive
for file_name in files_to_move:
    if os.path.exists(file_name):
        destination = os.path.join(drive_path, file_name)
        shutil.copy(file_name, destination)
        print(f"Successfully saved {file_name} to Drive.")
    else:
        print(f"Error: {file_name} not found in local environment.")

print("\nBackup Complete. Your Decision Support module is now stored in Google Drive.")

Mounted at /content/drive
Created new directory: /content/drive/MyDrive/AgriBot_Project/Models/
Successfully saved fertilizer_model.pkl to Drive.
Successfully saved le_crop_fert.pkl to Drive.
Error: le_stage_fert.pkl not found in local environment.

Backup Complete. Your Decision Support module is now stored in Google Drive.
